# LeetCode 355 — Design Twitter bằng Python

Link đề: https://leetcode.com/problems/design-twitter/ · Độ khó: **Medium** · Chủ đề: Design (thiết kế lớp), Hash Table, Linked List / danh sách, Heap (Priority Queue), gộp k danh sách đã sắp

Skeleton LeetCode:

```python
class Twitter:

    def __init__(self):
        ...

    def postTweet(self, userId: int, tweetId: int) -> None:
        ...

    def getNewsFeed(self, userId: int) -> List[int]:
        ...

    def follow(self, followerId: int, followeeId: int) -> None:
        ...

    def unfollow(self, followerId: int, followeeId: int) -> None:
        ...


# Your Twitter object will be instantiated and called as such:
# obj = Twitter()
# obj.postTweet(userId,tweetId)
# param_2 = obj.getNewsFeed(userId)
# obj.follow(followerId,followeeId)
# obj.unfollow(followerId,followeeId)
```

Ghi chú: ví dụ lấy từ đề; mọi cách giải trong notebook đã được kiểm chứng bằng code (đối chiếu bốn cách + một "oracle" độc lập trên hàng nghìn chuỗi lệnh ngẫu nhiên). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **bài thiết kế lớp**: không có một hàm duy nhất, mà phải thiết kế **trạng thái** (dữ liệu lưu trong `self`) rồi cài **5 thao tác** (`__init__`, `postTweet`, `getNewsFeed`, `follow`, `unfollow`); biết LeetCode "gọi" lớp bằng **chuỗi lệnh** (danh sách tên hàm + danh sách tham số) và tự dựng bộ chạy chuỗi lệnh để test.
2. Nắm các công cụ Python: `dict`/`defaultdict(list)`/`defaultdict(set)`, `set.add` / `set.discard` (khác `remove`), `list.append`, cắt đuôi `lst[-10:]`, `reversed`, `sorted(key=...)`, `heapq` (`heappush`, `heappop`, `heapify`, `nlargest`) và mẹo **heap max bằng số âm**.
3. Đi từ cách **quét toàn bộ tweet** → **gom tweet của mọi người đã theo dõi rồi sắp xếp** → **chỉ lấy 10 tweet mới nhất mỗi người** → **gộp k danh sách đã sắp bằng heap** (giống "merge k sorted lists").
4. Hiểu mấu chốt thiết kế: dùng **bộ đếm thời gian toàn cục** (`self.time`) để biết tweet nào mới hơn (đừng dùng `tweetId` làm thời gian!), mỗi người giữ danh sách tweet **theo thứ tự đăng** (đã sắp sẵn theo thời gian) nên có thể **gộp lười** bằng heap; `follow`/`unfollow` chỉ sửa một `set` nên `O(1)`.
5. Tránh các lỗi kinh điển: dùng `tweetId` để so sánh độ mới, quên đưa **chính mình** vào news feed, `unfollow` người chưa follow gây `KeyError`, sửa nhầm tập `following` thật khi thêm chính mình, và `heappop` trên heap rỗng khi feed ngắn hơn `10`.

## 2. Đề bài & ví dụ

Thiết kế một phiên bản **Twitter đơn giản**: người dùng có thể **đăng tweet**, **follow/unfollow** người khác, và xem được **10 tweet gần nhất** trong **news feed** của mình.

Cài đặt lớp `Twitter`:

- `Twitter()`: khởi tạo đối tượng.
- `postTweet(userId, tweetId)`: người dùng `userId` đăng tweet có ID `tweetId`. Mỗi lời gọi sẽ có `tweetId` **duy nhất**.
- `getNewsFeed(userId)`: trả về danh sách **tối đa 10** `tweetId` gần nhất trong news feed của `userId`. Mỗi tweet trong feed phải do **người mà `userId` đang follow** hoặc **chính `userId`** đăng. Thứ tự: **từ mới nhất đến cũ nhất**.
- `follow(followerId, followeeId)`: `followerId` bắt đầu follow `followeeId`.
- `unfollow(followerId, followeeId)`: `followerId` bỏ follow `followeeId`.

**Ví dụ 1**

```
Input
["Twitter", "postTweet", "getNewsFeed", "follow", "postTweet", "getNewsFeed", "unfollow", "getNewsFeed"]
[[], [1, 5], [1], [1, 2], [2, 6], [1], [1, 2], [1]]
Output
[null, null, [5], null, null, [6, 5], null, [5]]
```

Giải thích: user 1 đăng tweet 5 → feed `[5]`; user 1 follow user 2; user 2 đăng tweet 6 → feed của user 1 là `[6, 5]` (6 mới hơn nên đứng trước 5); user 1 unfollow user 2 → feed trở lại `[5]`.

**Ràng buộc:**

- `1 <= userId, followerId, followeeId <= 500`
- `0 <= tweetId <= 10^4`
- Mọi tweet có ID **khác nhau**.
- Tối đa `3 * 10^4` lời gọi tới `postTweet`, `getNewsFeed`, `follow`, `unfollow`.
- Một người **không thể follow chính mình**.

In [1]:
import random
import time
from collections import defaultdict
from heapq import heapify, heappop, heappush, nlargest
from typing import List

# Ví dụ 1 của đề, viết đúng dạng LeetCode: danh sách tên lệnh + danh sách tham số
EX_OPS = ["Twitter", "postTweet", "getNewsFeed", "follow", "postTweet", "getNewsFeed", "unfollow", "getNewsFeed"]
EX_ARGS = [[], [1, 5], [1], [1, 2], [2, 6], [1], [1, 2], [1]]
EX_OUT = [None, None, [5], None, None, [6, 5], None, [5]]
print(len(EX_OPS), "lệnh;", "kết quả mong đợi:", EX_OUT)

8 lệnh; kết quả mong đợi: [None, None, [5], None, None, [6, 5], None, [5]]


## 3. Kiến thức nền

### 3.1. LeetCode "chạy" một bài thiết kế như thế nào?

Bài này **không** có một hàm `Solution.xxx` cho input → output. LeetCode tạo một đối tượng, rồi gọi lần lượt các phương thức theo **hai danh sách**: `ops` (tên lệnh, phần tử đầu là tên lớp) và `args` (tham số của từng lệnh). Kết quả trả về là danh sách trong đó hàm `void` cho `null` (Python: `None`).

Để tự test, ta viết **bộ chạy** `run_commands(cls, ops, args)` mô phỏng đúng việc đó: phần tử đầu → tạo `cls(*args[0])`; phần còn lại → `getattr(obj, tên)(*tham_số)`.

In [2]:
def run_commands(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế lớp; trả về danh sách kết quả (None cho hàm void)."""
    obj = None
    out = []
    for name, a in zip(ops, args):
        if obj is None:                       # lệnh đầu tiên luôn là tên lớp -> khởi tạo
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, name)(*a))   # getattr(obj, "postTweet") lấy ra phương thức theo tên chuỗi
    return out


# thử bộ chạy với một lớp đồ chơi
class Counter:
    def __init__(self):
        self.x = 0
    def add(self, k):
        self.x += k
    def get(self):
        return self.x

print(run_commands(Counter, ["Counter", "add", "add", "get"], [[], [2], [3], []]))   # [None, None, None, 5]

[None, None, None, 5]


### 3.2. `dict`, `defaultdict` và `set`

Ta cần "với mỗi người dùng, nhớ danh sách tweet của họ" và "với mỗi người dùng, nhớ tập người họ follow". `dict` ánh xạ khóa → giá trị. `defaultdict(list)` tự tạo `[]` khi truy cập khóa chưa có; `defaultdict(set)` tự tạo `set()`.

`set` là tập không trùng, kiểm tra thuộc `in` mất `O(1)`. Hai cách xóa phần tử:

- `s.remove(x)`: nếu `x` **không có** → báo **`KeyError`**;
- `s.discard(x)`: nếu `x` không có → **im lặng bỏ qua** (an toàn cho `unfollow` người chưa follow).

In [3]:
tweets_of = defaultdict(list)
tweets_of[1].append(5)
tweets_of[1].append(7)
print(dict(tweets_of), tweets_of[99])      # truy cập khóa 99 chưa có -> tự tạo []
print(len(tweets_of), "(khóa 99 đã bị tạo ra!)")

following = defaultdict(set)
following[1].add(2)
following[1].add(2)                         # thêm trùng: không đổi
print(following[1])                         # {2}
following[1].discard(3)                     # an toàn dù 3 không có
try:
    following[1].remove(3)
except KeyError as e:
    print("remove(3) báo KeyError:", e)

{1: [5, 7]} []
2 (khóa 99 đã bị tạo ra!)
{2}
remove(3) báo KeyError: 3


### 3.3. Danh sách: `append`, cắt đuôi, đảo ngược

Mỗi người giữ danh sách tweet theo **thứ tự đăng** (cũ → mới), nên tweet mới nhất ở **cuối**. Cần 10 tweet mới nhất của họ → `lst[-10:]` (nếu ít hơn 10 phần tử thì lấy tất cả, không báo lỗi). Duyệt từ mới → cũ: `reversed(lst)` hoặc `lst[::-1]`.

In [4]:
lst = [10, 20, 30, 40, 50]
print(lst[-3:])              # [30, 40, 50]: 3 phần tử cuối
print(lst[-10:])             # [10, 20, 30, 40, 50]: ít hơn 10 thì lấy hết
print(lst[::-1])             # [50, 40, 30, 20, 10]
print(list(reversed(lst)))   # giống trên, nhưng reversed() là lười (iterator)
print([][-10:])              # []: danh sách rỗng cũng an toàn

[30, 40, 50]
[10, 20, 30, 40, 50]
[50, 40, 30, 20, 10]
[50, 40, 30, 20, 10]
[]


### 3.4. `sorted` với `key`, và `heapq.nlargest`

`sorted(items, key=f, reverse=True)` sắp theo giá trị `f(item)` giảm dần, tốn `O(m log m)`. `heapq.nlargest(k, items, key=f)` lấy `k` phần tử có `f` lớn nhất (đã xếp giảm dần), tốn khoảng `O(m log k)`.

In [5]:
items = [(3, "c"), (1, "a"), (4, "d"), (2, "b")]       # (thời gian, tweet)
print(sorted(items, key=lambda p: p[0], reverse=True)[:2])
print(nlargest(2, items, key=lambda p: p[0]))

[(4, 'd'), (3, 'c')]
[(4, 'd'), (3, 'c')]


### 3.5. `heapq` — heap nhỏ nhất và mẹo heap lớn nhất

`heapq` trong Python là **min-heap**: `heap[0]` luôn là phần tử **nhỏ nhất**.

- `heappush(h, x)`: thêm `x`, `O(log n)`;
- `heappop(h)`: lấy và trả phần tử nhỏ nhất, `O(log n)`;
- `heapify(list)`: biến list bất kỳ thành heap trong `O(n)`.

Muốn lấy phần tử **lớn nhất** (ở đây: tweet **mới nhất** = thời gian **lớn nhất**) thì lưu **số âm** (`-time`). Phần tử là tuple thì so sánh **từng thành phần từ trái sang phải**.

In [6]:
h = []
for t in [3, 8, 1, 6]:
    heappush(h, -t)                 # lưu -t để "nhỏ nhất" thực ra là t lớn nhất
print("lớn nhất:", -h[0])           # 8
print([-heappop(h) for _ in range(4)])   # [8, 6, 3, 1]

h2 = [(-5, "x"), (-9, "y"), (-7, "z")]
heapify(h2)
print(heappop(h2))                  # (-9, 'y'): tuple so sánh thành phần đầu trước

lớn nhất: 8
[8, 6, 3, 1]
(-9, 'y')


### 3.6. Bộ đếm thời gian — vì sao KHÔNG dùng `tweetId` để so độ mới?

Đề chỉ nói `tweetId` **duy nhất**, **không** nói chúng tăng dần. Tweet đăng sau có thể có `tweetId` nhỏ hơn (ví dụ đăng `tweetId = 9000` rồi mới đăng `tweetId = 3`). Vì vậy ta giữ `self.time` tăng thêm `1` mỗi lần `postTweet`, và gắn "dấu thời gian" đó cho tweet. Tweet nào có `time` lớn hơn thì mới hơn.

In [7]:
log = []                 # (thời gian, tweetId) theo thứ tự đăng
clock = 0
for tid in [9000, 3, 500]:
    clock += 1
    log.append((clock, tid))
print(log)
print("mới nhất theo time:", max(log)[1], "| mới nhất theo tweetId (SAI):", max(t for _, t in log))

[(1, 9000), (2, 3), (3, 500)]
mới nhất theo time: 500 | mới nhất theo tweetId (SAI): 9000


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay **Ví dụ 1** và ghi trạng thái sau mỗi lệnh (`time` là đồng hồ toàn cục):

| # | lệnh | tweets (theo người) | following | time | trả về |
|---|---|---|---|---|---|
| 1 | `postTweet(1, 5)` | `1: [(t1, 5)]` | — | 1 | — |
| 2 | `getNewsFeed(1)` | không đổi | — | 1 | `[5]` |
| 3 | `follow(1, 2)` | không đổi | `1: {2}` | 1 | — |
| 4 | `postTweet(2, 6)` | `1: [(t1, 5)]`, `2: [(t2, 6)]` | `1: {2}` | 2 | — |
| 5 | `getNewsFeed(1)` | không đổi | `1: {2}` | 2 | người cần xét `{1, 2}`: tweet `(t1,5)`, `(t2,6)` → mới nhất trước: `[6, 5]` |
| 6 | `unfollow(1, 2)` | không đổi | `1: {}` | 2 | — |
| 7 | `getNewsFeed(1)` | không đổi | `1: {}` | 2 | người cần xét `{1}` → `[5]` |

Trực giác về **news feed**: "tập người cần xét" = **những người mình follow + chính mình**; lấy **mọi tweet** của họ; xếp theo thời gian giảm dần; cắt `10` cái đầu. Bốn lệnh còn lại gần như chỉ **ghi dữ liệu**; chỗ khó (và nơi các cách khác nhau) là **cách lấy 10 tweet mới nhất** cho nhanh.

In [8]:
# Mô phỏng thô đúng như bảng trên (chưa phải lớp), để thấy "dữ liệu" thay đổi ra sao
tweets_of = defaultdict(list)       # userId -> [(time, tweetId), ...]
follows = defaultdict(set)          # userId -> {followeeId, ...}
clock = 0

def post(u, tid):
    global clock
    clock += 1
    tweets_of[u].append((clock, tid))

def feed(u):
    people = follows[u] | {u}                       # người đang follow + chính mình
    allt = [p for v in people for p in tweets_of[v]]
    allt.sort(reverse=True)                         # time lớn trước
    return [tid for _, tid in allt[:10]]

post(1, 5);            print(feed(1))               # [5]
follows[1].add(2)
post(2, 6);            print(feed(1))               # [6, 5]
follows[1].discard(2); print(feed(1))               # [5]

[5]
[6, 5]
[5]


## 5. Cách 1 — Brute Force / baseline: quét toàn bộ tweet từ mới đến cũ

**Ý tưởng.** Giữ **một nhật ký chung** `self.tweets` các tweet theo thứ tự đăng (cuối là mới nhất), mỗi phần tử là `(userId, tweetId)`; và `self.following[u]` là tập người `u` follow.

- `postTweet`: `append` vào cuối nhật ký.
- `follow` / `unfollow`: `add` / `discard` trên tập.
- `getNewsFeed(u)`: duyệt nhật ký **từ cuối về đầu**; tweet nào do `u` hoặc người `u` follow đăng thì lấy; dừng khi đủ `10`.

Dễ nhất để nghĩ ra vì nhật ký đã **tự sắp theo thời gian** — không cần so sánh gì.

In [9]:
class TwitterScan:
    """Cách 1: một nhật ký chung, quét từ cuối về đầu."""

    def __init__(self):
        self.tweets = []                       # (userId, tweetId) theo thứ tự đăng
        self.following = defaultdict(set)      # userId -> tập người đang follow

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.tweets.append((userId, tweetId))

    def getNewsFeed(self, userId: int) -> List[int]:
        people = self.following[userId] | {userId}
        feed = []
        for i in range(len(self.tweets) - 1, -1, -1):    # từ mới nhất về cũ nhất
            owner, tid = self.tweets[i]
            if owner in people:
                feed.append(tid)
                if len(feed) == 10:
                    break
        return feed

    def follow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].discard(followeeId)


print(run_commands(TwitterScan, EX_OPS, EX_ARGS))
assert run_commands(TwitterScan, EX_OPS, EX_ARGS) == EX_OUT

[None, None, [5], None, None, [6, 5], None, [5]]


**Độ phức tạp (Cách 1).** Gọi `T` là tổng số tweet đã đăng.

- `postTweet`, `follow`, `unfollow`: **`O(1)`**.
- `getNewsFeed`: **`O(T)`** trong trường hợp xấu (ví dụ người này chỉ có vài tweet và follow ít người, trong khi hàng nghìn tweet của người lạ nằm giữa — phải lội qua tất cả mới đủ/biết là không đủ `10`).
- Space: **`O(T + số cặp follow)`**.

**Nút thắt:** `getNewsFeed` phải đi qua cả tweet của **người không liên quan**. Với `3 * 10^4` lời gọi, `T` lên tới hàng chục nghìn và nhiều lần `getNewsFeed`, có thể thành `~10^8` bước — chậm. Ta nên **tổ chức dữ liệu theo từng người** để chỉ chạm vào tweet của những người liên quan.

In [10]:
# Minh họa nút thắt: user 1 không follow ai, không có tweet; user 2 đăng 6000 tweet; user 1 xem feed 1500 lần
def stress_scan(cls, tweets=6000, feeds=1500):
    obj = cls()
    for i in range(tweets):
        obj.postTweet(2, i)
    t0 = time.perf_counter()
    for _ in range(feeds):
        obj.getNewsFeed(1)
    return time.perf_counter() - t0

print(f"Cách 1 (quét hết): {stress_scan(TwitterScan):.2f} giây cho 1500 lần getNewsFeed (feed rỗng)")

Cách 1 (quét hết): 0.34 giây cho 1500 lần getNewsFeed (feed rỗng)


## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. **Tổ chức theo từng người.** Mỗi người `u` giữ danh sách tweet **của riêng họ**, mỗi tweet là cặp `(time, tweetId)`, xếp theo thứ tự đăng (tức `time` tăng dần). Khi cần feed của `u`, chỉ đụng tới danh sách của **`u` và những người `u` follow** — tweet người lạ không bao giờ bị chạm.
2. **Chỉ cần 10 tweet mới nhất của mỗi người.** News feed chỉ có 10 phần tử. Nếu một người đăng 1000 tweet, chỉ **10 tweet cuối** của họ có thể lọt vào top 10 của bất kỳ ai (vì top-10 chung chỉ lấy nhiều nhất 10 từ một nguồn). Càng tiến hơn: các danh sách đã **sắp sẵn theo thời gian**, nên đây là bài **gộp `k` danh sách đã sắp** (*merge k sorted lists*) — dùng **heap** chỉ giữ **một "con trỏ" cho mỗi người**: lấy cái mới nhất trong các con trỏ, rồi nhích con trỏ đó lùi một bước. Chỉ làm đúng **10 lần**.

Chi phí tăng dần: gom hết + sort → gom 10 cuối mỗi người + sort → heap gộp lười (không đụng tới tweet thừa).

## 7. Các cách tiếp theo

### Cách 2 — Danh sách theo từng người, gom **tất cả** tweet rồi sắp xếp

**Giải thích từng đoạn.**

1. `self.time`: đồng hồ toàn cục tăng `1` mỗi `postTweet`.
2. `self.tweets[u]`: danh sách `(time, tweetId)` của `u`, tăng dần theo `time` vì ta luôn `append`.
3. `getNewsFeed(u)`: `people = following[u] | {u}` (**lưu ý** toán tử `|` tạo `set` **mới**, không sửa `following[u]`); gom mọi cặp của mọi người trong `people`, `sort(reverse=True)` (so tuple → `time` lớn trước), lấy `10` cái đầu, trả về `tweetId`.

In [11]:
class TwitterSortAll:
    """Cách 2: danh sách theo từng người; gom hết rồi sắp xếp."""

    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)        # userId -> [(time, tweetId), ...] tăng dần theo time
        self.following = defaultdict(set)

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.time += 1
        self.tweets[userId].append((self.time, tweetId))

    def getNewsFeed(self, userId: int) -> List[int]:
        people = self.following[userId] | {userId}       # set mới: không đụng vào following[userId]
        pool = []
        for u in people:
            pool.extend(self.tweets[u])
        pool.sort(reverse=True)                          # time lớn (mới) trước
        return [tid for _, tid in pool[:10]]

    def follow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].discard(followeeId)


assert run_commands(TwitterSortAll, EX_OPS, EX_ARGS) == EX_OUT
print(run_commands(TwitterSortAll, EX_OPS, EX_ARGS))

[None, None, [5], None, None, [6, 5], None, [5]]


Code nộp LeetCode cho Cách 2: đổi tên `TwitterSortAll` thành `Twitter` (ở trên đã đủ 5 hàm đúng skeleton).

**Độ phức tạp (Cách 2).** Gọi `M` là tổng số tweet của `u` và những người `u` follow. `getNewsFeed`: **`O(M log M)`**; các hàm còn lại `O(1)`. Space: `O(T + số cặp follow)`. Tốt hơn Cách 1 vì không đụng tweet người lạ, nhưng nếu `u` follow người đăng rất nhiều tweet thì vẫn phải sort cả đống.

### Cách 3 — Chỉ lấy **10 tweet cuối** của mỗi người rồi sắp xếp

**Giải thích.** Cách 2 chỉ đổi đúng một chỗ: thay vì `extend(self.tweets[u])` (cả danh sách), ta `extend(self.tweets[u][-10:])` — **10 phần tử cuối** (mới nhất) của người đó. Lý do đúng: top-10 của feed không thể chứa tweet thứ 11 trở đi (tính từ mới nhất) của một người, vì 10 tweet mới hơn của chính người ấy đã đứng trước nó. Pool giờ tối đa `10·F` phần tử (`F` = số người xét) → dùng `nlargest(10, pool)` hoặc `sort`.

In [12]:
class TwitterLast10:
    """Cách 3: mỗi người chỉ đóng góp 10 tweet mới nhất; sắp xếp pool nhỏ."""

    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)
        self.following = defaultdict(set)

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.time += 1
        self.tweets[userId].append((self.time, tweetId))

    def getNewsFeed(self, userId: int) -> List[int]:
        people = self.following[userId] | {userId}
        pool = []
        for u in people:
            pool.extend(self.tweets[u][-10:])           # 10 tweet cuối = 10 tweet mới nhất của u
        top = nlargest(10, pool)                         # so tuple -> time lớn nhất; đã xếp giảm dần
        return [tid for _, tid in top]

    def follow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].discard(followeeId)


assert run_commands(TwitterLast10, EX_OPS, EX_ARGS) == EX_OUT
print(run_commands(TwitterLast10, EX_OPS, EX_ARGS))

[None, None, [5], None, None, [6, 5], None, [5]]


**Độ phức tạp (Cách 3).** Với `F = |following| + 1`: pool có `<= 10F` phần tử; `nlargest(10, pool)` tốn `O(10F · log 10)` ≈ `O(F)`. Vậy `getNewsFeed` ≈ **`O(F)`** (hằng số `10`); gom mỗi `lst[-10:]` tốn `O(10)`. Space `O(T + số cặp follow)` cho dữ liệu, `O(F)` tạm thời.

### Cách 4 — Gộp `k` danh sách đã sắp bằng **heap** (cách chuẩn, lười)

**Giải thích từng đoạn.**

1. Mỗi người có danh sách `(time, tweetId)` tăng dần theo `time`. Tweet mới nhất của người đó ở chỉ số `len - 1`.
2. Heap giữ **một phần tử cho mỗi người** (con trỏ hiện tại): `(-time, tweetId, userId, index)`. Dùng `-time` để `heappop` trả tweet **mới nhất** trong các con trỏ.
3. Lặp tối đa `10` lần: `heappop` ra phần tử có `time` lớn nhất → thêm `tweetId` vào feed; nếu người đó còn tweet cũ hơn (`index > 0`) thì `heappush` tweet ở `index - 1` của người đó.
4. Dừng khi đủ `10` hoặc heap rỗng (feed ít hơn `10`).
5. Khởi tạo heap: thêm con trỏ cuối của từng người có tweet, rồi `heapify` (`O(F)`) — chỉ cần `F` phần tử đầu, **không** đụng tới tweet thừa.

Vì `time` luôn **khác nhau** nên các tuple không bao giờ so sánh tới `tweetId` (nhưng có cũng không hại).

In [13]:
class TwitterHeap:
    """Cách 4: gộp lười k danh sách bằng heap."""

    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)        # userId -> [(time, tweetId), ...] tăng dần theo time
        self.following = defaultdict(set)

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.time += 1
        self.tweets[userId].append((self.time, tweetId))

    def getNewsFeed(self, userId: int) -> List[int]:
        people = self.following[userId] | {userId}
        heap = []
        for u in people:
            lst = self.tweets[u]
            if lst:                                           # người chưa đăng gì: bỏ qua
                i = len(lst) - 1                              # con trỏ vào tweet mới nhất
                heap.append((-lst[i][0], lst[i][1], u, i))
        heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, tid, u, i = heappop(heap)                      # tweet mới nhất trong các con trỏ
            feed.append(tid)
            if i > 0:                                         # nhích con trỏ của u lùi một tweet
                t2, tid2 = self.tweets[u][i - 1]
                heappush(heap, (-t2, tid2, u, i - 1))
        return feed

    def follow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].discard(followeeId)


assert run_commands(TwitterHeap, EX_OPS, EX_ARGS) == EX_OUT
print(run_commands(TwitterHeap, EX_OPS, EX_ARGS))

[None, None, [5], None, None, [6, 5], None, [5]]


**Độ phức tạp (Cách 4).** `getNewsFeed`: dựng heap `O(F)`, rồi tối đa `10` lần `heappop` + `heappush` mỗi lần `O(log F)` ⇒ **`O(F + 10 log F)`** = `O(F)` với `10` là hằng. Các hàm còn lại `O(1)`. Space: `O(T + số cặp follow)` cho dữ liệu; `O(F)` heap tạm.

So với Cách 3: cùng bậc `O(F)` nhưng Cách 4 **không bao giờ** nhìn tới tweet vượt quá nhu cầu (rất đẹp nếu mỗi người có hàng nghìn tweet), và là mẫu hình quen thuộc "gộp `k` danh sách đã sắp" (LeetCode 23, 373, 378...).

Chạy lại bài stress nhỏ để thấy sự khác nhau về tốc độ so với Cách 1.

In [14]:
for cls in (TwitterScan, TwitterSortAll, TwitterLast10, TwitterHeap):
    print(f"{cls.__name__:15s}: {stress_scan(cls):.3f} giây")
# Cách 1 chậm vì phải quét 6000 tweet của người lạ ở mỗi lần; các cách còn lại chỉ đụng tới danh sách của user 1 (rỗng)

TwitterScan    : 0.292 giây
TwitterSortAll : 0.000 giây
TwitterLast10  : 0.001 giây
TwitterHeap    : 0.000 giây


## 8. TRACE từng bước

### 8.1. Trace `getNewsFeed` bằng heap (Cách 4)

Kịch bản (tự dựng): có `3` người. `time` ghi trong ngoặc.

- user 1 đăng: `10 (t1)`, `11 (t3)`, `12 (t6)`
- user 2 đăng: `20 (t2)`, `21 (t5)`
- user 3 đăng: `30 (t4)`
- user 1 follow `2` và `3`.

Danh sách theo người: `1: [(1,10), (3,11), (6,12)]`, `2: [(2,20), (5,21)]`, `3: [(4,30)]`. Feed của user 1 (mới → cũ) phải là `[12, 21, 30, 11, 20, 10]`, vì thứ tự thời gian: `t6, t5, t4, t3, t2, t1`.

| lần | heap (trước pop) — `(-time, tweetId, user, idx)` | pop | feed | đẩy thêm |
|---|---|---|---|---|
| 1 | `(-6,12,1,2) (-5,21,2,1) (-4,30,3,0)` | `(-6,12,1,2)` | `[12]` | `(-3,11,1,1)` |
| 2 | `(-5,21,2,1) (-4,30,3,0) (-3,11,1,1)` | `(-5,21,2,1)` | `[12,21]` | `(-2,20,2,0)` |
| 3 | `(-4,30,3,0) (-3,11,1,1) (-2,20,2,0)` | `(-4,30,3,0)` | `[12,21,30]` | không (idx = 0) |
| 4 | `(-3,11,1,1) (-2,20,2,0)` | `(-3,11,1,1)` | `[12,21,30,11]` | `(-1,10,1,0)` |
| 5 | `(-2,20,2,0) (-1,10,1,0)` | `(-2,20,2,0)` | `[...,20]` | không |
| 6 | `(-1,10,1,0)` | `(-1,10,1,0)` | `[...,10]` | không; heap rỗng → dừng |

Hàm `trace_feed` dưới đây in đúng các bước đó.

In [15]:
def trace_feed(tw, userId):
    """In từng vòng của getNewsFeed theo Cách 4 trên đối tượng tw (kiểu TwitterHeap)."""
    people = tw.following[userId] | {userId}
    print("người cần xét:", sorted(people))
    heap = []
    for u in people:
        lst = tw.tweets[u]
        if lst:
            i = len(lst) - 1
            heap.append((-lst[i][0], lst[i][1], u, i))
    heapify(heap)
    feed = []
    step = 0
    while heap and len(feed) < 10:
        step += 1
        print(f"  vòng {step}: heap = {sorted(heap)}")
        _, tid, u, i = heappop(heap)
        feed.append(tid)
        msg = "không đẩy thêm (hết tweet của user %d)" % u
        if i > 0:
            t2, tid2 = tw.tweets[u][i - 1]
            heappush(heap, (-t2, tid2, u, i - 1))
            msg = f"đẩy (-{t2}, {tid2}, {u}, {i - 1})"
        print(f"          pop tweet {tid} của user {u} -> feed = {feed}; {msg}")
    print("kết quả:", feed)
    return feed


tw = TwitterHeap()
for u, tid in [(1, 10), (2, 20), (1, 11), (3, 30), (2, 21), (1, 12)]:     # time lần lượt 1..6
    tw.postTweet(u, tid)
tw.follow(1, 2)
tw.follow(1, 3)
res = trace_feed(tw, 1)
assert res == [12, 21, 30, 11, 20, 10]

người cần xét: [1, 2, 3]
  vòng 1: heap = [(-6, 12, 1, 2), (-5, 21, 2, 1), (-4, 30, 3, 0)]
          pop tweet 12 của user 1 -> feed = [12]; đẩy (-3, 11, 1, 1)
  vòng 2: heap = [(-5, 21, 2, 1), (-4, 30, 3, 0), (-3, 11, 1, 1)]
          pop tweet 21 của user 2 -> feed = [12, 21]; đẩy (-2, 20, 2, 0)
  vòng 3: heap = [(-4, 30, 3, 0), (-3, 11, 1, 1), (-2, 20, 2, 0)]
          pop tweet 30 của user 3 -> feed = [12, 21, 30]; không đẩy thêm (hết tweet của user 3)
  vòng 4: heap = [(-3, 11, 1, 1), (-2, 20, 2, 0)]
          pop tweet 11 của user 1 -> feed = [12, 21, 30, 11]; đẩy (-1, 10, 1, 0)
  vòng 5: heap = [(-2, 20, 2, 0), (-1, 10, 1, 0)]
          pop tweet 20 của user 2 -> feed = [12, 21, 30, 11, 20]; không đẩy thêm (hết tweet của user 2)
  vòng 6: heap = [(-1, 10, 1, 0)]
          pop tweet 10 của user 1 -> feed = [12, 21, 30, 11, 20, 10]; không đẩy thêm (hết tweet của user 1)
kết quả: [12, 21, 30, 11, 20, 10]


### 8.2. Trace toàn bộ Ví dụ 1 qua lớp (in trạng thái sau mỗi lệnh)

Hàm `trace_commands` chạy từng lệnh và in `time`, `tweets`, `following` để bạn đối chiếu với bảng ở mục 4.

In [16]:
def trace_commands(cls, ops, args):
    obj = None
    for step, (name, a) in enumerate(zip(ops, args), 1):
        if obj is None:
            obj = cls(*a)
            print(f"{step}. tạo {cls.__name__}()")
            continue
        res = getattr(obj, name)(*a)
        print(f"{step}. {name}{tuple(a)} -> {res}")
        print(f"     time={obj.time}  tweets={ {u: v for u, v in obj.tweets.items() if v} }  "
              f"following={ {u: s for u, s in obj.following.items() if s} }")


trace_commands(TwitterHeap, EX_OPS, EX_ARGS)

1. tạo TwitterHeap()
2. postTweet(1, 5) -> None
     time=1  tweets={1: [(1, 5)]}  following={}
3. getNewsFeed(1,) -> [5]
     time=1  tweets={1: [(1, 5)]}  following={}
4. follow(1, 2) -> None
     time=1  tweets={1: [(1, 5)]}  following={1: {2}}
5. postTweet(2, 6) -> None
     time=2  tweets={1: [(1, 5)], 2: [(2, 6)]}  following={1: {2}}
6. getNewsFeed(1,) -> [6, 5]
     time=2  tweets={1: [(1, 5)], 2: [(2, 6)]}  following={1: {2}}
7. unfollow(1, 2) -> None
     time=2  tweets={1: [(1, 5)], 2: [(2, 6)]}  following={}
8. getNewsFeed(1,) -> [5]
     time=2  tweets={1: [(1, 5)], 2: [(2, 6)]}  following={}


## 9. Code nộp LeetCode

Mình nộp **Cách 4 (heap gộp lười)**: dễ chứng minh, nhanh nhất ở trường hợp xấu, và là mẫu "merge k sorted lists" đáng nhớ. Đã `import` đủ thư viện (một số môi trường LeetCode đã nạp sẵn nhưng viết rõ cho chắc).

In [17]:
from collections import defaultdict
from heapq import heapify, heappop, heappush
from typing import List


class Twitter:

    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)        # userId -> [(time, tweetId), ...] tăng dần theo time
        self.following = defaultdict(set)      # userId -> tập followeeId

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.time += 1
        self.tweets[userId].append((self.time, tweetId))

    def getNewsFeed(self, userId: int) -> List[int]:
        people = self.following[userId] | {userId}      # tập mới; không sửa following[userId]
        heap = []
        for u in people:
            lst = self.tweets[u]
            if lst:
                i = len(lst) - 1
                heap.append((-lst[i][0], lst[i][1], u, i))
        heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, tid, u, i = heappop(heap)
            feed.append(tid)
            if i > 0:
                t2, tid2 = self.tweets[u][i - 1]
                heappush(heap, (-t2, tid2, u, i - 1))
        return feed

    def follow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].discard(followeeId)


# Your Twitter object will be instantiated and called as such:
# obj = Twitter()
# obj.postTweet(userId,tweetId)
# param_2 = obj.getNewsFeed(userId)
# obj.follow(followerId,followeeId)
# obj.unfollow(followerId,followeeId)

out = run_commands(Twitter, EX_OPS, EX_ARGS)
print(out)
assert out == EX_OUT

[None, None, [5], None, None, [6, 5], None, [5]]


## 10. Vì sao thuật toán đúng

- **Bất biến dữ liệu:** mỗi `self.tweets[u]` luôn **tăng chặt** theo `time` vì ta chỉ `append` với `self.time` vừa tăng. `time` **toàn cục** nên so sánh `time` giữa **hai người khác nhau** có nghĩa (tweet nào đăng sau).
- **`follow` / `unfollow` đúng:** chúng chỉ thêm/bỏ phần tử trong `self.following[followerId]`. `discard` làm `unfollow` người chưa follow thành "không làm gì", đúng như đề cho phép. Follow hai lần = một lần (do `set`).
- **Tập người cần xét:** `self.following[u] | {u}` chính là "những người `u` follow **và** chính `u`", đúng định nghĩa news feed. Toán tử `|` tạo `set` mới nên `following[u]` không bị thêm `u`.
- **Heap gộp đúng thứ tự:** với `k` danh sách mỗi cái giảm dần (khi duyệt từ cuối về đầu), phần tử lớn nhất toàn cục luôn là **phần tử đầu** của một trong các danh sách — tức là một trong các con trỏ trong heap. `heappop` lấy đúng con trỏ có `time` lớn nhất ⇒ tweet mới nhất chưa được lấy. Sau khi lấy, ứng viên kế tiếp từ cùng danh sách đó là phần tử ngay trước nó, ta đẩy vào heap để bất biến "heap chứa phần tử đầu còn lại của mỗi danh sách" được giữ.
- **Dừng:** sau tối đa `10` lần lấy, hoặc khi heap rỗng (đã hết mọi tweet ⇒ feed ngắn hơn `10`), kết quả là `<= 10` tweet theo thứ tự mới → cũ, đúng đề.

## 11. Phân tích Time / Space Complexity

Gọi `T` = tổng số tweet, `F` = số người cần xét (`|following| + 1`, tối đa `500`), `K = 10`.

| Cách | `postTweet` | `follow`/`unfollow` | `getNewsFeed` | Space |
|---|---|---|---|---|
| 1. Quét nhật ký chung | `O(1)` | `O(1)` | `O(T)` | `O(T + P)` |
| 2. Gom hết + sort | `O(1)` | `O(1)` | `O(M log M)`, `M` = tweet của các người xét | `O(T + P)` |
| 3. 10 tweet cuối mỗi người + `nlargest` | `O(1)` | `O(1)` | `O(K·F)` | `O(T + P)` |
| 4. Heap gộp lười | `O(1)` | `O(1)` | `O(F + K log F)` | `O(T + P)` |

(`P` = số cặp follow đang có; vì có tối đa `3 * 10^4` lời gọi nên `P <= 3 * 10^4`.)

Với giới hạn của đề: `F <= 500`, `<= 3 * 10^4` lời gọi ⇒ tệ nhất khoảng `3 * 10^4 · 500 = 1.5 * 10^7` bước cho Cách 3/4 — nhanh. Cách 1 và 2 có thể chậm hơn nhiều khi một số người đăng hàng chục nghìn tweet.

## 12. Bộ test

Với bài thiết kế, "test" là **chuỗi lệnh** và **kết quả mong đợi** cho từng lệnh. Ta dùng `run_commands` rồi `assert` trên mọi cách. Ví dụ đề đã chạy ở trên; bây giờ thêm các trường hợp biên và đối chiếu ngẫu nhiên.

In [18]:
ALL = [TwitterScan, TwitterSortAll, TwitterLast10, TwitterHeap, Twitter]


def check_all(ops, args, expected):
    for cls in ALL:
        got = run_commands(cls, ops, args)
        assert got == expected, (cls.__name__, got, expected)


# 1) Ví dụ đề
check_all(EX_OPS, EX_ARGS, EX_OUT)

# 2) Người chưa đăng gì, chưa follow ai -> feed rỗng
check_all(["Twitter", "getNewsFeed"], [[], [7]], [None, []])

# 3) Feed bị cắt ở 10, thứ tự mới -> cũ: đăng 15 tweet (id 100..114) -> chỉ 114..105
ops = ["Twitter"] + ["postTweet"] * 15 + ["getNewsFeed"]
args = [[]] + [[1, 100 + i] for i in range(15)] + [[1]]
check_all(ops, args, [None] * 16 + [list(range(114, 104, -1))])

# 4) Đúng 10 tweet và 11 tweet
ops = ["Twitter"] + ["postTweet"] * 10 + ["getNewsFeed", "postTweet", "getNewsFeed"]
args = [[]] + [[1, i] for i in range(10)] + [[1], [1, 10], [1]]
check_all(ops, args, [None] * 11 + [list(range(9, -1, -1))] + [None] + [list(range(10, 0, -1))])

# 5) unfollow người chưa follow: không lỗi, không đổi gì
check_all(["Twitter", "postTweet", "unfollow", "getNewsFeed"], [[], [1, 5], [1, 2], [1]], [None, None, None, [5]])

# 6) follow hai lần rồi unfollow một lần: hết follow (set)
ops = ["Twitter", "postTweet", "follow", "follow", "unfollow", "getNewsFeed"]
args = [[], [2, 9], [1, 2], [1, 2], [1, 2], [1]]
check_all(ops, args, [None, None, None, None, None, []])

# 7) tweetId KHÔNG tăng dần: thứ tự theo thời gian đăng, không theo tweetId
ops = ["Twitter", "postTweet", "postTweet", "postTweet", "getNewsFeed"]
args = [[], [1, 9000], [1, 3], [1, 500], [1]]
check_all(ops, args, [None, None, None, None, [500, 3, 9000]])

# 8) tweetId = 0 hợp lệ (đừng dùng "if tweetId" để kiểm tra tồn tại)
check_all(["Twitter", "postTweet", "getNewsFeed"], [[], [1, 0], [1]], [None, None, [0]])

# 9) Follow rồi unfollow rồi follow lại: tweet cũ của người đó lại hiện
ops = ["Twitter", "postTweet", "follow", "getNewsFeed", "unfollow", "getNewsFeed", "follow", "getNewsFeed"]
args = [[], [2, 40], [1, 2], [1], [1, 2], [1], [1, 2], [1]]
check_all(ops, args, [None, None, None, [40], None, [], None, [40]])

# 10) Hai người follow nhau, feed mỗi người thấy tweet của cả hai theo thứ tự thời gian
ops = ["Twitter", "follow", "follow", "postTweet", "postTweet", "postTweet", "getNewsFeed", "getNewsFeed"]
args = [[], [1, 2], [2, 1], [1, 1], [2, 2], [1, 3], [1], [2]]
check_all(ops, args, [None, None, None, None, None, None, [3, 2, 1], [3, 2, 1]])

# 11) User có hơn 10 tweet nhưng follow người có tweet xen kẽ
ops = ["Twitter", "follow"]
args = [[], [1, 2]]
tid = 0
for k in range(12):
    ops += ["postTweet", "postTweet"]
    args += [[1, tid], [2, tid + 1]]
    tid += 2
ops.append("getNewsFeed")
args.append([1])
exp = [None] * (len(ops) - 1) + [list(range(23, 13, -1))]
check_all(ops, args, exp)
print("Các test cơ bản đều qua")

Các test cơ bản đều qua


### 12.2. Phản ví dụ cho các cách làm sai

Dưới đây là hai lớp **sai** để thấy test có "bắt" được lỗi hay không:

- `BadByTweetId`: lấy `tweetId` lớn nhất làm "mới nhất".
- `BadNoSelf`: quên đưa **chính mình** vào feed.

In [19]:
class BadByTweetId:
    """SAI: so sánh theo tweetId thay vì thời gian đăng."""
    def __init__(self):
        self.tweets = defaultdict(list)
        self.following = defaultdict(set)
        self.time = 0
    def postTweet(self, userId, tweetId):
        self.tweets[userId].append(tweetId)
    def getNewsFeed(self, userId):
        people = self.following[userId] | {userId}
        pool = [t for u in people for t in self.tweets[u]]
        return sorted(pool, reverse=True)[:10]        # sắp theo tweetId giảm dần
    def follow(self, a, b):
        self.following[a].add(b)
    def unfollow(self, a, b):
        self.following[a].discard(b)


class BadNoSelf(TwitterSortAll):
    """SAI: feed không có tweet của chính mình."""
    def getNewsFeed(self, userId):
        pool = []
        for u in self.following[userId]:
            pool.extend(self.tweets[u])
        pool.sort(reverse=True)
        return [tid for _, tid in pool[:10]]


ops = ["Twitter", "postTweet", "postTweet", "getNewsFeed"]
args = [[], [1, 9000], [1, 3], [1]]
print("đúng        :", run_commands(Twitter, ops, args)[-1])          # [3, 9000]
print("BadByTweetId:", run_commands(BadByTweetId, ops, args)[-1])     # [9000, 3]  <- sai
print("BadNoSelf   :", run_commands(BadNoSelf, ops, args)[-1])        # []         <- sai
assert run_commands(BadByTweetId, ops, args) != run_commands(Twitter, ops, args)
assert run_commands(BadNoSelf, ops, args) != run_commands(Twitter, ops, args)

đúng        : [3, 9000]
BadByTweetId: [9000, 3]
BadNoSelf   : []


### 12.3. Đối chiếu ngẫu nhiên với một "oracle" độc lập (có seed)

Oracle `OracleTwitter` cố ý cài **theo cách khác hẳn**: nó không dùng heap hay danh sách theo người; nó chỉ ghi `stamp[tweetId] = thứ tự đăng`, `owner[tweetId]`, và mỗi lần hỏi feed thì **lọc + sort** theo `stamp`. Sinh chuỗi lệnh ngẫu nhiên **đúng ràng buộc**: `userId` trong `[1, U]` (`U` nhỏ để dễ va chạm), `tweetId` duy nhất (lấy từ `random.sample` trong `[0, 10^4]` nên **không** tăng dần), không follow chính mình.

In [20]:
class OracleTwitter:
    def __init__(self):
        self.stamp = {}                  # tweetId -> số thứ tự đăng
        self.owner = {}                  # tweetId -> userId
        self.edges = set()               # {(follower, followee)}
        self.count = 0

    def postTweet(self, userId, tweetId):
        self.count += 1
        self.stamp[tweetId] = self.count
        self.owner[tweetId] = userId

    def getNewsFeed(self, userId):
        ok = [t for t in self.owner if self.owner[t] == userId or (userId, self.owner[t]) in self.edges]
        ok.sort(key=lambda t: self.stamp[t], reverse=True)
        return ok[:10]

    def follow(self, a, b):
        self.edges.add((a, b))

    def unfollow(self, a, b):
        self.edges.discard((a, b))


def random_commands(rng, n_ops, n_users, post_weight=4):
    ids = rng.sample(range(0, 10001), n_ops)          # tweetId duy nhất, thứ tự ngẫu nhiên
    ops, args = ["Twitter"], [[]]
    for k in range(n_ops):
        kind = rng.choices(["postTweet", "getNewsFeed", "follow", "unfollow"], [post_weight, 3, 2, 1])[0]
        if kind == "postTweet":
            ops.append(kind); args.append([rng.randint(1, n_users), ids[k]])
        elif kind == "getNewsFeed":
            ops.append(kind); args.append([rng.randint(1, n_users)])
        else:
            a = rng.randint(1, n_users)
            b = rng.randint(1, n_users)
            while b == a:
                b = rng.randint(1, n_users)
            ops.append(kind); args.append([a, b])
    return ops, args


rng = random.Random(355)
total_feed_calls = 0
for case in range(600):
    n_users = rng.choice([2, 3, 4, 6, 10])
    n_ops = rng.randint(1, 120)
    ops, args = random_commands(rng, n_ops, n_users, post_weight=rng.choice([2, 4, 8]))
    expected = run_commands(OracleTwitter, ops, args)
    for cls in ALL:
        got = run_commands(cls, ops, args)
        assert got == expected, (cls.__name__, case)
    total_feed_calls += ops.count("getNewsFeed")
print("600 chuỗi lệnh ngẫu nhiên, tổng", total_feed_calls, "lần getNewsFeed: mọi cách khớp oracle")

# Ca lớn: 10^4 lệnh (tweetId duy nhất trong [0, 10^4] nên tối đa ~10^4 tweet), 500 người; đối chiếu Cách 3 và Cách 4
rng = random.Random(3550)
ops, args = random_commands(rng, 10000, 500, post_weight=5)
t0 = time.perf_counter()
r3 = run_commands(TwitterLast10, ops, args)
r4 = run_commands(Twitter, ops, args)
assert r3 == r4
print(f"ca lớn khớp nhau ({len(ops)} lệnh) trong {time.perf_counter() - t0:.2f} giây")

600 chuỗi lệnh ngẫu nhiên, tổng 11318 lần getNewsFeed: mọi cách khớp oracle
ca lớn khớp nhau (10001 lệnh) trong 0.02 giây


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `tweetId` để so độ mới

**Code sai.** Nghĩ "ID lớn = mới hơn":
```python
return sorted(pool, reverse=True)[:10]      # pool toàn là tweetId
```
**Vì sao sai.** Đề chỉ nói `tweetId` duy nhất, không nói tăng dần. Đăng `9000` rồi đăng `3` → tweet `3` mới hơn nhưng `3 < 9000`.

**Code đúng.** Gắn `time` toàn cục tăng dần cho mỗi tweet và so theo `time`.

In [21]:
ops = ["Twitter", "postTweet", "postTweet", "getNewsFeed"]
args = [[], [1, 9000], [1, 3], [1]]
print("sai   :", run_commands(BadByTweetId, ops, args)[-1])   # [9000, 3]
print("đúng  :", run_commands(Twitter, ops, args)[-1])        # [3, 9000]

sai   : [9000, 3]
đúng  : [3, 9000]


### Lỗi 2: Quên đưa chính mình vào feed

**Code sai.**
```python
for u in self.following[userId]:      # chỉ người mình follow
    ...
```
**Vì sao sai.** Đề: feed gồm tweet của người đã follow **hoặc chính người dùng**. Ví dụ 1: user 1 chưa follow ai vẫn thấy tweet `5` của mình.

**Code đúng.** `people = self.following[userId] | {userId}`.

In [22]:
print("sai  :", run_commands(BadNoSelf, EX_OPS, EX_ARGS))
print("đúng :", run_commands(Twitter, EX_OPS, EX_ARGS))

sai  : [None, None, [], None, None, [6], None, []]
đúng : [None, None, [5], None, None, [6, 5], None, [5]]


### Lỗi 3: `unfollow` bằng `remove` → `KeyError`

**Code sai.**
```python
def unfollow(self, followerId, followeeId):
    self.following[followerId].remove(followeeId)
```
**Vì sao sai.** Đề không cấm `unfollow` người chưa follow (và test của LeetCode có thể làm vậy). `set.remove` báo `KeyError` khi phần tử không có.

**Code đúng.** `discard` (hoặc `if followeeId in s: s.remove(followeeId)`).

In [23]:
class BadRemove(TwitterSortAll):
    def unfollow(self, followerId, followeeId):
        self.following[followerId].remove(followeeId)       # KeyError nếu chưa follow


try:
    run_commands(BadRemove, ["Twitter", "unfollow"], [[], [1, 2]])
except KeyError as e:
    print("sai: KeyError", e)
print("đúng:", run_commands(Twitter, ["Twitter", "unfollow"], [[], [1, 2]]))     # [None, None]

sai: KeyError 2
đúng: [None, None]


### Lỗi 4: Sửa nhầm tập `following` thật khi thêm chính mình

**Code sai.**
```python
people = self.following[userId]       # KHÔNG phải bản sao: cùng một đối tượng set!
people.add(userId)                    # làm following[userId] chứa luôn chính mình
```
**Vì sao sai.** `people` và `self.following[userId]` là **cùng một `set`**. Sau lệnh `add`, người dùng "tự follow mình" vĩnh viễn — vi phạm ràng buộc và có thể gây sai ở các hàm khác (ví dụ đếm số người đang follow).

**Code đúng.** `people = self.following[userId] | {userId}` (tạo `set` mới) hoặc `people = set(self.following[userId]); people.add(userId)`.

In [24]:
class BadAlias(TwitterSortAll):
    def getNewsFeed(self, userId):
        people = self.following[userId]          # alias!
        people.add(userId)
        pool = []
        for u in people:
            pool.extend(self.tweets[u])
        pool.sort(reverse=True)
        return [tid for _, tid in pool[:10]]


bad = BadAlias(); good = Twitter()
for obj in (bad, good):
    obj.postTweet(1, 5)
    obj.getNewsFeed(1)
print("sai : following[1] =", dict(bad.following)[1])      # {1}: tự follow mình!
print("đúng: following[1] =", dict(good.following).get(1, set()))

sai : following[1] = {1}
đúng: following[1] = set()


### Lỗi 5: `heappop` khi heap rỗng (feed ngắn hơn 10)

**Code sai.**
```python
for _ in range(10):
    _, tid, u, i = heappop(heap)       # IndexError nếu chỉ có 3 tweet
```
**Vì sao sai.** Người dùng có thể có ít hơn `10` tweet trong feed.

**Code đúng.** Lặp `while heap and len(feed) < 10`.

In [25]:
def feed_bad_loop(tw, userId):
    heap = []
    for u in tw.following[userId] | {userId}:
        lst = tw.tweets[u]
        if lst:
            heap.append((-lst[-1][0], lst[-1][1], u, len(lst) - 1))
    heapify(heap)
    feed = []
    for _ in range(10):                      # SAI: không kiểm tra heap rỗng
        _, tid, u, i = heappop(heap)
        feed.append(tid)
        if i > 0:
            t2, tid2 = tw.tweets[u][i - 1]
            heappush(heap, (-t2, tid2, u, i - 1))
    return feed


tw = TwitterHeap(); tw.postTweet(1, 5); tw.postTweet(1, 6)
try:
    feed_bad_loop(tw, 1)
except IndexError as e:
    print("sai: IndexError:", e)
print("đúng:", tw.getNewsFeed(1))

sai: IndexError: index out of range
đúng: [6, 5]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "**thiết kế lớp**" ⇒ liệt kê **thao tác** và nghĩ: dữ liệu nào cần lưu để mỗi thao tác nhanh? Thường: một `dict` (khóa → thông tin) + một cấu trúc phụ (set, heap, list).
- "Top `k` mới nhất từ nhiều nguồn đã sắp theo thời gian" ⇒ **merge k sorted lists** bằng **heap** (một con trỏ/nguồn). Gặp lại ở: LeetCode 23 (Merge k Sorted Lists), 378, 373, 264.
- "Cái nào đăng sau?" ⇒ dùng **đồng hồ tăng dần** thay vì dựa vào ID.
- `defaultdict(list)` / `defaultdict(set)` rút gọn code, nhưng nhớ: **truy cập** khóa chưa có cũng **tạo** khóa (ảnh hưởng `len`, duyệt).
- `follow`/`unfollow` luôn dùng `set` + `discard` — vừa `O(1)` vừa không lỗi.
- Heap max trong Python = lưu **số âm**; tuple so sánh từng thành phần, nên đặt **khóa so sánh ở đầu** (`-time`) và các trường khác sau.
- Test thiết kế: viết `run_commands` ngay từ đầu, rồi gõ chuỗi lệnh dạng LeetCode; so với một oracle chậm nhưng hiển nhiên đúng.
- Đừng tối ưu sớm: với giới hạn `500` người, `3 * 10^4` lệnh, Cách 3 hay 4 đều đủ nhanh.

## 15. Biến thể của bài

1. **`getNewsFeed` trả `k` tweet** (thay vì cố định `10`): chỉ cần đổi hằng `10` thành tham số.
2. **Push model (fan-out khi ghi):** mỗi người giữ sẵn **feed riêng**; khi `postTweet`, đẩy tweet vào feed của **mọi follower** (đọc nhanh `O(1)`, ghi chậm `O(số follower)`); `follow`/`unfollow` phải cập nhật feed. Hệ thống thật (Twitter/X) dùng kết hợp push + pull.
3. **Xóa tweet** (`deleteTweet(userId, tweetId)`): cần xóa khỏi danh sách theo người (lazy delete bằng tập `deleted`, bỏ qua khi duyệt).
4. **Phân trang / cuộn tiếp:** `getNewsFeed(userId, cursor)` trả 10 tweet tiếp theo — heap gộp lười rất hợp vì có thể tiếp tục từ trạng thái đã lưu.
5. **`getFollowers`, `getFollowees`, đếm lượt like...** — chỉ thêm cấu trúc dữ liệu phụ.
6. **Bài cùng họ "thiết kế":** LeetCode 146 (LRU Cache), 380 (Insert Delete GetRandom O(1)), 295 (Find Median from Data Stream), 460 (LFU Cache).

In [26]:
class TwitterK(Twitter):
    """Biến thể: số tweet trong feed là tham số k."""

    def getNewsFeedK(self, userId: int, k: int) -> List[int]:
        people = self.following[userId] | {userId}
        heap = []
        for u in people:
            lst = self.tweets[u]
            if lst:
                heap.append((-lst[-1][0], lst[-1][1], u, len(lst) - 1))
        heapify(heap)
        feed = []
        while heap and len(feed) < k:
            _, tid, u, i = heappop(heap)
            feed.append(tid)
            if i > 0:
                t2, tid2 = self.tweets[u][i - 1]
                heappush(heap, (-t2, tid2, u, i - 1))
        return feed


tk = TwitterK()
for tid in range(1, 8):
    tk.postTweet(1, tid)
print(tk.getNewsFeedK(1, 3), tk.getNewsFeedK(1, 20))
assert tk.getNewsFeedK(1, 3) == [7, 6, 5]
assert tk.getNewsFeedK(1, 20) == [7, 6, 5, 4, 3, 2, 1]
assert tk.getNewsFeedK(1, 10) == tk.getNewsFeed(1)

[7, 6, 5] [7, 6, 5, 4, 3, 2, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | `getNewsFeed` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét nhật ký chung | một list chung, duyệt từ mới → cũ | `O(T)` | `O(T + P)` | hiểu đề, dùng làm oracle; ít tweet |
| 2. Gom hết + sort | list theo người, gom rồi sort | `O(M log M)` | `O(T + P)` | ít tweet mỗi người |
| 3. 10 cuối mỗi người + `nlargest` | chỉ lấy 10 mới nhất/người | `O(10·F)` | `O(T + P)` | code ngắn, đủ nhanh cho đề |
| 4. Heap gộp lười | một con trỏ/người, pop 10 lần | `O(F + 10 log F)` | `O(T + P)` | **nên nộp**, mẫu "merge k sorted lists" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán kết quả `getNewsFeed(1)` sau chuỗi lệnh: `postTweet(1, 7)`, `postTweet(2, 3)`, `follow(1, 2)`, `postTweet(2, 9)`, `postTweet(1, 4)`, rồi `unfollow(1, 2)`, `getNewsFeed(1)`. Cũng cho biết `getNewsFeed(1)` **trước** lệnh `unfollow`.

**Bài 2 (tính tay).** Với `time` bắt đầu từ `0`, hãy viết các cặp `(time, tweetId)` của từng người và các bước `heappop` khi lấy feed của user `1` (có follow `2`): user 1 đăng `5, 8`; user 2 đăng `6`; user 1 đăng `2`; user 2 đăng `1, 7` (theo thứ tự thời gian này). Liệt kê heap và feed sau mỗi lần pop.

**Bài 3 (điền TODO).** Hoàn thành `feed_last10_todo(tweets, following, userId)` theo Cách 3: nhận `tweets` (dict userId → list `(time, tweetId)`), `following` (dict userId → set), trả về feed.

In [27]:
def feed_last10_todo(tweets, following, userId):
    # TODO 1: people = following.get(userId, set()) | {userId}
    # TODO 2: pool = []; với mỗi u trong people: pool.extend(tweets.get(u, [])[-10:])
    # TODO 3: lấy 10 phần tử lớn nhất của pool (theo time) và trả về danh sách tweetId
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết lại **từ đầu** lớp `MyTwitter` theo Cách 4 (heap gộp lười) mà **không nhìn** code mẫu, rồi kiểm tra bằng `run_commands(MyTwitter, EX_OPS, EX_ARGS) == EX_OUT`.

**Bài 5 (biến thể / mini challenge).** Thêm hai hàm vào lớp: `getFollowers(userId)` trả **tập** những người đang follow `userId`, và `getNewsFeedWithOwner(userId)` trả danh sách `(owner, tweetId)` (tối đa 10, mới → cũ). Gợi ý: cần lưu thêm `owner` trong heap/tuple; `getFollowers` nên được duy trì sẵn hay tính mỗi lần? Hãy cài cả hai rồi so tốc độ trên dữ liệu 500 người.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: thứ tự đăng là `7 (user 1)`, `3 (user 2)`, `9 (user 2)`, `4 (user 1)`; trước `unfollow` feed của 1 gồm cả hai người, xếp mới → cũ: `[4, 9, 3, 7]`; sau `unfollow` chỉ còn tweet của user 1: `[4, 7]`. Bài 3: ba dòng đúng theo ba TODO; dùng `nlargest(10, pool)`.
- **Mức 2 (vừa):** Bài 2: thứ tự đăng là `(1,5)`, `(2,8)` của user 1, `(3,6)` của user 2, `(4,2)` của user 1, `(5,1)`, `(6,7)` của user 2. Heap khởi đầu là phần tử cuối mỗi người: user 1 → `(-4,2)`, user 2 → `(-6,7)`. Pop `(-6,7)` trước, đẩy `(-5,1)`...
- **Mức 3 (sâu):** Bài 4: chia thành `postTweet` (tăng `time`, append), `follow/unfollow` (`add/discard`), `getNewsFeed` (heap + while). Bài 5: dùng thêm `defaultdict(set)` `followers`; trong `follow` cập nhật cả hai tập; `getNewsFeedWithOwner` giống Cách 4 nhưng đưa `u` vào kết quả.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1: thứ tự đăng: (t1: 1,7) (t2: 2,3) (t3: 2,9) (t4: 1,4)
t = Twitter()
for args in [(1, 7), (2, 3)]:
    t.postTweet(*args)
t.follow(1, 2)
t.postTweet(2, 9)
t.postTweet(1, 4)
assert t.getNewsFeed(1) == [4, 9, 3, 7]        # trước unfollow: t4, t3, t2, t1
t.unfollow(1, 2)
assert t.getNewsFeed(1) == [4, 7]              # sau unfollow chỉ còn tweet của user 1

# Bài 2
t = Twitter()
for u, tid in [(1, 5), (1, 8), (2, 6), (1, 2), (2, 1), (2, 7)]:
    t.postTweet(u, tid)
t.follow(1, 2)
assert t.tweets[1] == [(1, 5), (2, 8), (4, 2)]
assert t.tweets[2] == [(3, 6), (5, 1), (6, 7)]
assert t.getNewsFeed(1) == [7, 1, 2, 6, 8, 5]       # time 6,5,4,3,2,1


# Bài 3
def feed_last10_todo_ans(tweets, following, userId):
    people = following.get(userId, set()) | {userId}
    pool = []
    for u in people:
        pool.extend(tweets.get(u, [])[-10:])
    return [tid for _, tid in nlargest(10, pool)]


assert feed_last10_todo_ans({1: [(1, 5), (2, 8), (4, 2)], 2: [(3, 6), (5, 1), (6, 7)]}, {1: {2}}, 1) == [7, 1, 2, 6, 8, 5]
assert feed_last10_todo_ans({}, {}, 3) == []


# Bài 4
class MyTwitter:
    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)
        self.following = defaultdict(set)

    def postTweet(self, userId, tweetId):
        self.time += 1
        self.tweets[userId].append((self.time, tweetId))

    def getNewsFeed(self, userId):
        heap = []
        for u in self.following[userId] | {userId}:
            lst = self.tweets[u]
            if lst:
                heap.append((-lst[-1][0], lst[-1][1], u, len(lst) - 1))
        heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, tid, u, i = heappop(heap)
            feed.append(tid)
            if i > 0:
                t2, tid2 = self.tweets[u][i - 1]
                heappush(heap, (-t2, tid2, u, i - 1))
        return feed

    def follow(self, a, b):
        self.following[a].add(b)

    def unfollow(self, a, b):
        self.following[a].discard(b)


assert run_commands(MyTwitter, EX_OPS, EX_ARGS) == EX_OUT


# Bài 5
class TwitterPlus(MyTwitter):
    def __init__(self):
        super().__init__()
        self.followers = defaultdict(set)

    def follow(self, a, b):
        super().follow(a, b)
        self.followers[b].add(a)

    def unfollow(self, a, b):
        super().unfollow(a, b)
        self.followers[b].discard(a)

    def getFollowers(self, userId):
        return set(self.followers[userId])

    def getNewsFeedWithOwner(self, userId):
        heap = []
        for u in self.following[userId] | {userId}:
            lst = self.tweets[u]
            if lst:
                heap.append((-lst[-1][0], lst[-1][1], u, len(lst) - 1))
        heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, tid, u, i = heappop(heap)
            feed.append((u, tid))
            if i > 0:
                t2, tid2 = self.tweets[u][i - 1]
                heappush(heap, (-t2, tid2, u, i - 1))
        return feed


tp = TwitterPlus()
tp.postTweet(1, 5); tp.follow(2, 1); tp.follow(3, 1); tp.postTweet(2, 6); tp.follow(2, 3)
assert tp.getFollowers(1) == {2, 3}
tp.unfollow(3, 1)
assert tp.getFollowers(1) == {2}
assert tp.getNewsFeedWithOwner(2) == [(2, 6), (1, 5)]
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (heap gộp lười)

```python
from collections import defaultdict
from heapq import heapify, heappop, heappush
from typing import List


class Twitter:
    def __init__(self):
        self.time = 0
        self.tweets = defaultdict(list)       # u -> [(time, tweetId)] tăng dần
        self.following = defaultdict(set)     # u -> {followee}

    def postTweet(self, userId: int, tweetId: int) -> None:
        self.time += 1
        self.tweets[userId].append((self.time, tweetId))

    def getNewsFeed(self, userId: int) -> List[int]:
        heap = []
        for u in self.following[userId] | {userId}:
            lst = self.tweets[u]
            if lst:
                heap.append((-lst[-1][0], lst[-1][1], u, len(lst) - 1))
        heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, tid, u, i = heappop(heap)
            feed.append(tid)
            if i > 0:
                t2, tid2 = self.tweets[u][i - 1]
                heappush(heap, (-t2, tid2, u, i - 1))
        return feed

    def follow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].add(followeeId)

    def unfollow(self, followerId: int, followeeId: int) -> None:
        self.following[followerId].discard(followeeId)
```

**Checklist**

- [ ] Có **đồng hồ toàn cục** `time`, không dùng `tweetId` để so độ mới?
- [ ] Feed gồm tweet của **người mình follow và chính mình** (`| {userId}`)?
- [ ] Dùng `discard` (không `remove`) cho `unfollow`; `follow` bằng `set.add`?
- [ ] Không sửa nhầm `self.following[userId]` khi thêm chính mình (tạo `set` mới)?
- [ ] Vòng lấy feed là `while heap and len(feed) < 10` (không `for _ in range(10)` rỗng heap)?
- [ ] Heap lưu `-time` đầu tuple; khi pop xong có đẩy tweet cũ hơn của **cùng** người đó?
- [ ] Trả `list` (không phải tuple/generator), thứ tự mới → cũ, tối đa `10`?
- [ ] Đã thử: ví dụ đề, feed rỗng, > 10 tweet, unfollow người chưa follow, `tweetId = 0`, `tweetId` không tăng dần?
- [ ] Skeleton đúng `class Twitter` với 5 phương thức và chữ ký như đề?